In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics.pairwise import euclidean_distances, cosine_distances
from sklearn.feature_extraction.text import TfidfVectorizer
!pip install nltk
import nltk
from nltk.corpus import stopwords
import re


[notice] A new release of pip is available: 25.0 -> 25.1
[notice] To update, run: pip install --upgrade pip


In [2]:
review_data = pd.read_excel('/Users/qianxinhui/Desktop/NUSTAT/415/Hw 2 & 3: Recommender Systems/Evanston Restaurant Reviews.xlsx')
review_data.head()

,Restaurant Name,Cuisine,Latitude,Longitude,Average Cost,Open After 8pm?,Brief Description
0,Tapas Barcelona,Spanish,42.046736,-87.679043,20,Yes,"Festive, warm space known for Spanish small pl..."
1,Lao Sze Chuan,Chinese,42.048462,-87.679476,20,Yes,"Modern Chinese mainstay, known for an extensiv..."
2,5411 Empanadas,Spanish,42.047310,-87.681849,13,Yes,Known for Argentinean empanadas & special sauc...
3,Hokkaido Ramen,Japanese,42.048482,-87.682722,13,Yes,"Whimsical ramen bar, known for sushi rolls and..."
4,Tomo Japanese Street Food,Japanese,42.049612,-87.682046,20,Yes,Japanese street food cafe with mobile app orde...


In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Load the data
# Adjust file path as needed
file_path = '/Users/qianxinhui/Desktop/NUSTAT/415/Hw 2 & 3: Recommender Systems/Evanston Restaurant Reviews.xlsx'
restaurants = pd.read_excel(file_path, sheet_name="Restaurants")
reviews = pd.read_excel(file_path, sheet_name="Reviews")

# Display missing values before handling
print("Missing values in restaurants before handling:")
print(restaurants.isnull().sum())
print("\nMissing values in reviews before handling:")
print(reviews.isnull().sum())

# ------------------------
# HANDLING MISSING VALUES
# ------------------------

# 1. Review Text (38.27% missing)
# For text data, we'll fill with a placeholder
if 'Review Text' in reviews.columns:
    reviews['Review Text'] = reviews['Review Text'].fillna("[No review provided]")

# 2. Vegetarian? (93.67% missing)
# Given extreme missingness, create a new feature indicating known status
if 'Vegetarian?' in reviews.columns:
    # Create indicator feature
    reviews['Vegetarian_Known'] = reviews['Vegetarian?'].notna().astype(int)
    # Preserve original data where available and default to "No" elsewhere
    reviews['Vegetarian?'] = reviews['Vegetarian?'].fillna('No')

# 3. Weight and Height (moderate missingness)
# Use median imputation for these numerical values
for col in ['Weight (lb)', 'Height (cm)']:
    if col in reviews.columns and reviews[col].isnull().sum() > 0:
        median_val = reviews[col].median()
        reviews[col] = reviews[col].fillna(median_val)
        print(f"Imputed {col} with median value: {median_val}")

# 4. Birth Year, Marital Status, Has Children? (low missingness)
# Use mode imputation for these demographic variables
if 'Birth Year' in reviews.columns:
    mode_year = reviews['Birth Year'].mode()[0]
    reviews['Birth Year'] = reviews['Birth Year'].fillna(mode_year)
    print(f"Imputed Birth Year with mode: {mode_year}")

# For categorical variables with low missingness
for col in ['Marital Status', 'Has Children?']:
    if col in reviews.columns and reviews[col].isnull().sum() > 0:
        mode_val = reviews[col].mode()[0]
        reviews[col] = reviews[col].fillna(mode_val)
        print(f"Imputed {col} with mode: {mode_val}")

# 5. Preferred Mode of Transport
if 'Preferred Mode of Transport' in reviews.columns:
    mode_transport = reviews['Preferred Mode of Transport'].mode()[0]
    reviews['Preferred Mode of Transport'] = reviews['Preferred Mode of Transport'].fillna(mode_transport)
    print(f"Imputed Preferred Mode of Transport with mode: {mode_transport}")

# 6. Average Amount Spent
if 'Average Amount Spent' in reviews.columns:
    # Use more sophisticated imputation for this important feature
    # KNN imputation based on other user features
    relevant_cols = ['Birth Year']
    # Add other columns if they exist and have low missingness
    for col in ['Has Children?', 'Marital Status', 'Preferred Mode of Transport']:
        if col in reviews.columns and reviews[col].isnull().sum() / len(reviews) < 0.1:
            # For categorical columns, we need to encode them first
            if reviews[col].dtype == 'object':
                dummies = pd.get_dummies(reviews[col], prefix=col, dummy_na=False)
                for dummy_col in dummies.columns:
                    reviews[dummy_col] = dummies[dummy_col]
                relevant_cols.extend(dummies.columns.tolist())
            else:
                relevant_cols.append(col)
    
    # Only use KNN imputation if we have enough relevant columns
    if len(relevant_cols) >= 2 and 'Average Amount Spent' in reviews.columns:
        # Subset data for KNN imputation
        imputation_df = reviews[relevant_cols + ['Average Amount Spent']].copy()
        # Fill any NaNs in the features first
        for col in relevant_cols:
            if imputation_df[col].isnull().sum() > 0:
                imputation_df[col] = imputation_df[col].fillna(imputation_df[col].median() 
                                                 if imputation_df[col].dtype != 'object' 
                                                 else imputation_df[col].mode()[0])
        
        # Apply KNN imputation
        imputer = KNNImputer(n_neighbors=5)
        imputed_vals = imputer.fit_transform(imputation_df)
        # Extract the imputed Average Amount Spent column
        reviews['Average Amount Spent'] = imputed_vals[:, -1]
        print("Imputed Average Amount Spent using KNN imputation")
    else:
        # Simple median imputation as fallback
        median_spent = reviews['Average Amount Spent'].median()
        reviews['Average Amount Spent'] = reviews['Average Amount Spent'].fillna(median_spent)
        print(f"Imputed Average Amount Spent with median: {median_spent}")

# For Restaurants data (minimal missingness reported)
# Simple imputation for any missing values
for col in restaurants.columns:
    if restaurants[col].isnull().sum() > 0:
        if restaurants[col].dtype == 'object':
            # Mode imputation for categorical
            restaurants[col] = restaurants[col].fillna(restaurants[col].mode()[0])
        else:
            # Median imputation for numerical
            restaurants[col] = restaurants[col].fillna(restaurants[col].median())

# ------------------------
# ADVANCED IMPUTATION (Optional)
# ------------------------

# MICE (Multiple Imputation by Chained Equations) for preserving relationships
# Uncomment this section if you want to use it for numerical columns
"""
if 'Birth Year' in reviews.columns and 'Weight (lb)' in reviews.columns and 'Height (cm)' in reviews.columns:
    # Select only numerical columns for MICE
    num_cols = ['Birth Year', 'Weight (lb)', 'Height (cm)', 'Average Amount Spent']
    num_cols = [col for col in num_cols if col in reviews.columns]
    
    if len(num_cols) >= 2:  # Need at least 2 columns for meaningful imputation
        # Create a subset with only numerical columns
        num_df = reviews[num_cols].copy()
        
        # Initialize and fit the iterative imputer
        mice_imputer = IterativeImputer(max_iter=10, random_state=42)
        imputed_data = mice_imputer.fit_transform(num_df)
        
        # Update the original dataframe with imputed values
        for i, col in enumerate(num_cols):
            reviews[col] = imputed_data[:, i]
        
        print("Applied MICE imputation for numerical columns to preserve relationships")
"""

# ------------------------
# VERIFICATION
# ------------------------

# Display missing values after handling
print("\nMissing values in restaurants after handling:")
print(restaurants.isnull().sum())
print("\nMissing values in reviews after handling:")
print(reviews.isnull().sum())

# Save the processed datasets
restaurants.to_csv('restaurants_processed.csv', index=False)
reviews.to_csv('reviews_processed.csv', index=False)

print("\nMissing values handled and preprocessed data saved to CSV files.")

Missing values in restaurants before handling:
Restaurant Name      0
Cuisine              0
Latitude             0
Longitude            0
Average Cost         0
Open After 8pm?      0
Brief Description    0
dtype: int64

Missing values in reviews before handling:
Reviewer Name                     0
Restaurant Name                   0
Rating                            0
Review Text                     574
Date of Review                    0
Birth Year                        2
Marital Status                   35
Has Children?                    38
Vegetarian?                    1405
Weight (lb)                      97
Height (cm)                      54
Average Amount Spent              1
Preferred Mode of Transport       4
Northwestern Student?             0
dtype: int64
Imputed Weight (lb) with median value: 200.0
Imputed Height (cm) with median value: 171.0
Imputed Birth Year with mode: 1999.0
Imputed Marital Status with mode: Single
Imputed Has Children? with mode: No
Imputed Prefer

ValueError: could not convert string to float: 'Medium'